# STED spot processing pipeline
### 0) Create projections
- Take 2 channel `.msr` images as input, located in `{in_path}/raw`.
- Create `{in_path}/projections` folder with 2 channel merged projections (red, green) `.png`.

### 1) Split channels and resave `msr` as `tif`
- Take 2 channel `.msr` images as input, located in `{in_path}/raw`.
- Create `{in_path}\tif` folder containg `.tif_ch{X}` of each input image.

### 2) Spot detection
- Use single-channel `.tif` and RS-FISH detection settings files to detect spots.
- Outputs a `{in_path}/{out_subfolder}/merge.csv` table of spots detected in each image.
- Outputs a `{in_path}/{out_subfolder}/vis` folder with `.png` max projections of spot detections.
- Look at the `spot-detection/RS-FISH_spot_detection.ipynb` notebook for more info

### 3) Get spot pairs
- Take dataframe of spots detected in STED detail images `spot_subfolder`.
- Combine the spots from channel 0 (eg. promoter) and channel 1 (eg. enhancer) in each image. Channel number is adjustable with `channels`.
- Calculate distances between each pair.
- Filter for images with exact number of enhnacers `n_enh` with max channel 0 - 1 distance `limit`.
- Save result to `merge_distances.csv`.
- Get list of "good" images (utilized in the `merge_distances.csv` dataframe).
- Link detection projections of those images into `projections_subfolder`.

### 4) (optional) Join all spots into a big csv

# 0) Imports and functions

In [6]:
import papermill as pm
import pandas as pd
import numpy as np
from multiprocessing import Pool
import concurrent.futures
import queue
import os
from pathlib import Path
from datetime import datetime

In [ ]:
def run_notebook(parameters,notebook_to_run,parameters_common={}):
    
    # change to directory where the notebook is (resolve relative imports)
    os.chdir(Path(notebook_to_run).absolute().parent)
    
    # run notebook
    for parameters_spec in parameters_list:
        parameters = {**parameters_common, **parameters_spec}

        pm.execute_notebook(
           notebook_to_run,
           '/dev/null',
           parameters=parameters)

# 0) Create projections

In [ ]:
parameters_list = [
    {"in_path": "/data/agl_data/NanoFISH/Gabi/GS550_Nanog_all_EpiSC/"}
]

notebook_to_run = "/home/stumberger/image-analyis-recipes/visualization/msr_make_projections_2_color.ipynb"

run_notebook(parameters_list,notebook_to_run)

# 1) Split channels and resave `msr` as `tif`

In [ ]:
parameters_list = [
    {"in_path": "/data/agl_data/NanoFISH/Gabi/GS550_Nanog_all_EpiSC/"}
]

notebook_to_run = "/home/stumberger/image-analyis-recipes/resave/resave_msr_as_tiff.ipynb"

run_notebook(parameters_list,notebook_to_run)

# 2) Spot detection

In [ ]:
parameters_list = [
     {"in_path": "/data/agl_data/NanoFISH/Gabi/GS550_Nanog_all_EpiSC/"}
]

parameters_common = {"channels": [0,1],
                     "tif_subfolder": "tif",
                    "out_subfolder": "detections"}

notebook_to_run = "/home/stumberger/image-analyis-recipes/spot-detection/RS-FISH_spot_detection-3d_vis.ipynb"

run_notebook(parameters_list,notebook_to_run,parameters_common)

# 3) Get spot pairs

In [ ]:
parameters_list = [
     {"in_path": "/data/agl_data/NanoFISH/Gabi/GS550_Nanog_all_EpiSC/"}
]

parameters_common = {"pixel_size": [0.03, 0.03, 0.015], # pixel sizes xyz [um]
                     "limit": 1.5, # max P-E distance to look at [um]
                     "n_enh": 3, # expected number of enhnacers
                     "channels": [0,1],
                     "spot_subfolder": "detections/merge.csv",
                     "projections_subfolder": "detections_good"}

notebook_to_run = "/home/stumberger/image-analyis-recipes/measurement/paired_spot_distances_2_channels_sted.ipynb"

run_notebook(parameters_list,notebook_to_run,parameters_common)

# 4) (optional) Join all spots into a big csv

In [ ]:
wd = "/data/agl_data/NanoFISH/Gabi/"

csv_files = [
    "/data/agl_data/NanoFISH/Gabi/GS464_Nanog_all_EpiSC/detections/",
    "/data/agl_data/NanoFISH/Gabi/GS463_Nanog_all_mESC/detections/",
    "/data/agl_data/NanoFISH/Gabi/GS491_Nanog_all_mESC/detections/",
    "/data/agl_data/NanoFISH/Gabi/GS492_Nanog_all_EpiSC/detections/",
    "/data/agl_data/NanoFISH/Gabi/GS549_Nanog_all_mESC/detections/",
    "/data/agl_data/NanoFISH/Gabi/GS550_Nanog_all_EpiSC/detections/",
    
]

# Initialize an empty list to store DataFrames
dataframes = []

# Read and store each CSV file as a DataFrame
for file in csv_files:
    df = pd.read_csv(f"{file}/merge_distances.csv")
    dataframes.append(df)

# Join the DataFrames using Pandas (e.g., concatenate them vertically)
joined_dataframe = pd.concat(dataframes, ignore_index=True)
# Save the joined DataFrame to a new CSV file
time =  datetime.now().strftime("%Y-%m-%d-%H-%M")
joined_dataframe.to_csv(f'{wd}/multi_sted_distances_{time}.csv', index=False)